# TEM-Inspired World Model Training
**Runtime → Change runtime type → A100 GPU** before running.

In [ ]:
# Setup
!git clone https://github.com/YX234/tem-generalization.git
%cd tem-generalization
!pip install -q gymnasium[mujoco] mujoco scikit-learn

In [ ]:
# Verify GPU
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Train
!python train.py

---
## Visualizations
Run these cells after training completes to inspect the learned representations.

In [ ]:
# Load trained model and normalizer
import glob, os, sys, pickle
import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA

sys.path.insert(0, '.')
from tem_model import TEMModel
from environment import DomainRandomizedHopper, RunningNormalizer

# Find latest checkpoint
model_dirs = sorted(glob.glob('runs/*/models'))
if not model_dirs:
    raise FileNotFoundError('No training runs found. Train the model first.')
model_dir = model_dirs[-1]
# Use final or latest numbered checkpoint
ckpt = os.path.join(model_dir, 'tem_final.pt')
if not os.path.exists(ckpt):
    ckpts = sorted(glob.glob(os.path.join(model_dir, 'tem_*.pt')))
    ckpt = ckpts[-1]
print(f'Loading: {ckpt}')

# Load saved config from checkpoint (must match the model architecture)
idx = os.path.basename(ckpt).replace('tem_', '').replace('.pt', '')
cfg_path = os.path.join(model_dir, f'cfg_{idx}.pt')
cfg = torch.load(cfg_path, weights_only=False)
print(f'Config loaded: g_dim={sum(cfg["n_g"])}, n_p={sum(cfg["n_p"])}')

# Load normalizer (must match the one used during training)
norm_path = os.path.join(model_dir, f'normalizer_{idx}.pkl')
if not os.path.exists(norm_path):
    norm_paths = sorted(glob.glob(os.path.join(model_dir, 'normalizer_*.pkl')))
    norm_path = norm_paths[-1] if norm_paths else None
if norm_path:
    with open(norm_path, 'rb') as f:
        normalizer = pickle.load(f)
    print(f'Normalizer loaded (trained on {normalizer.count} observations)')
else:
    print('WARNING: No normalizer found — using raw observations (results may be wrong)')
    normalizer = None

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = TEMModel(cfg, device=device).to(device)
model.load_state_dict(torch.load(ckpt, map_location=device, weights_only=True))
model.eval()
print(f'Model loaded on {device}')

def normalize_obs(obs_raw):
    """Normalize a single observation using the training normalizer."""
    if normalizer is not None:
        return normalizer.normalize(obs_raw)
    return obs_raw.astype(np.float32)

In [ ]:
# Collect representations from multiple domain-randomized episodes
n_episodes = 8
steps_per_ep = 200

all_g, all_obs, all_obs_raw, all_pred, all_ep, all_t, all_masses = [], [], [], [], [], [], []

for ep in range(n_episodes):
    cfg_eval = cfg.copy()
    cfg_eval['randomize'] = True
    env = DomainRandomizedHopper(cfg_eval, seed=5000 + ep)
    obs, _ = env.reset()
    total_mass = float(env.body_params['body_mass'].sum())

    chunk = []
    raw_obs_list = []
    for t in range(steps_per_ep):
        action = env.action_space.sample()
        obs_normed = normalize_obs(obs)
        raw_obs_list.append(obs.copy())
        chunk.append({
            'obs': torch.tensor(obs_normed, dtype=torch.float).unsqueeze(0).to(device),
            'action': torch.tensor(action, dtype=torch.float).unsqueeze(0).to(device),
        })
        obs, _, term, trunc, _ = env.step(action)
        if term or trunc:
            obs, _ = env.reset()
    env.close()

    with torch.no_grad():
        steps = model(chunk)

    for t, step in enumerate(steps):
        all_g.append(torch.cat(step.g_inf, dim=1).squeeze(0).cpu().numpy())
        all_obs.append(step.obs.squeeze(0).cpu().numpy())  # normalized
        all_obs_raw.append(raw_obs_list[t])
        all_pred.append(step.x_gen[0][0].squeeze(0).cpu().numpy())  # prediction (normalized space)
        all_ep.append(ep)
        all_t.append(t)
        all_masses.append(total_mass)

all_g = np.stack(all_g)
all_obs = np.stack(all_obs)
all_obs_raw = np.stack(all_obs_raw)
all_pred = np.stack(all_pred)
all_ep = np.array(all_ep)
all_t = np.array(all_t)
all_masses = np.array(all_masses)

print(f'Collected {len(all_g)} data points from {n_episodes} episodes')

### 1. Training Loss Curves

In [ ]:
# Parse training log for loss curves
import re

log_files = sorted(glob.glob('runs/*/train.log'))
if not log_files:
    print('No log file found')
else:
    iters, losses, mse_p, mse_ginf, mse_ggen = [], [], [], [], []
    with open(log_files[-1]) as f:
        for line in f:
            m = re.search(r'Iter\s+(\d+).*loss ([\-\d.]+).*mse\(p\) ([\d.]+) mse\(g_inf\) ([\d.]+) mse\(g_gen\) ([\d.]+)', line)
            if m:
                iters.append(int(m.group(1)))
                losses.append(float(m.group(2)))
                mse_p.append(float(m.group(3)))
                mse_ginf.append(float(m.group(4)))
                mse_ggen.append(float(m.group(5)))

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    ax = axes[0]
    ax.plot(iters, losses, alpha=0.3, color='blue')
    # Smoothed
    if len(losses) > 50:
        w = 50
        smoothed = np.convolve(losses, np.ones(w)/w, mode='valid')
        ax.plot(iters[w-1:], smoothed, color='blue', linewidth=2, label='smoothed')
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Total Loss')
    ax.set_title('Training Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)

    ax = axes[1]
    ax.plot(iters, mse_p, alpha=0.3, color='red')
    ax.plot(iters, mse_ginf, alpha=0.3, color='green')
    ax.plot(iters, mse_ggen, alpha=0.3, color='blue')
    if len(mse_p) > 50:
        for vals, color, label in [(mse_p, 'red', 'from p_inf'), (mse_ginf, 'green', 'from g_inf'), (mse_ggen, 'blue', 'from g_gen')]:
            smoothed = np.convolve(vals, np.ones(w)/w, mode='valid')
            ax.plot(iters[w-1:], smoothed, color=color, linewidth=2, label=label)
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Observation MSE')
    ax.set_title('Observation Prediction Accuracy')
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
    print(f'Final obs MSE: {mse_p[-1]:.4f} (from p), {mse_ginf[-1]:.4f} (from g_inf), {mse_ggen[-1]:.4f} (from g_gen)')

### 2. Observation Prediction Quality
How well does the model predict each observation dimension?

In [ ]:
dim_names = ['z_pos', 'angle', 'thigh_ang', 'leg_ang', 'foot_ang',
             'x_vel', 'z_vel', 'ang_vel', 'thigh_vel', 'leg_vel', 'foot_vel']

# MSE in normalized space (what the model optimizes)
mse_per_dim = np.mean((all_obs - all_pred) ** 2, axis=0)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Bar chart of per-dimension MSE (normalized space)
ax = axes[0]
colors = ['#2196F3' if mse < np.median(mse_per_dim) else '#FF5722' for mse in mse_per_dim]
ax.bar(range(len(dim_names)), mse_per_dim, color=colors)
ax.set_xticks(range(len(dim_names)))
ax.set_xticklabels(dim_names, rotation=45, ha='right')
ax.set_ylabel('MSE (normalized)')
ax.set_title(f'Prediction MSE per Dimension (total={np.mean(mse_per_dim):.4f})')
ax.grid(True, alpha=0.3, axis='y')

# Predicted vs actual scatter for best and worst dimensions
ax = axes[1]
best_dim = np.argmin(mse_per_dim)
worst_dim = np.argmax(mse_per_dim)
subset = np.random.choice(len(all_obs), min(500, len(all_obs)), replace=False)
ax.scatter(all_obs[subset, best_dim], all_pred[subset, best_dim], s=3, alpha=0.4, label=f'Best: {dim_names[best_dim]}')
ax.scatter(all_obs[subset, worst_dim], all_pred[subset, worst_dim], s=3, alpha=0.4, label=f'Worst: {dim_names[worst_dim]}')
lims = [min(ax.get_xlim()[0], ax.get_ylim()[0]), max(ax.get_xlim()[1], ax.get_ylim()[1])]
ax.plot(lims, lims, 'k--', alpha=0.3, label='Perfect')
ax.set_xlabel('Actual (normalized)')
ax.set_ylabel('Predicted (normalized)')
ax.set_title('Predicted vs Actual Observations')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 3. Abstract State Space (g) Visualization
Does g capture dynamical structure (gait phase, balance) rather than body-specific details?

In [ ]:
# t-SNE of g-space
print('Running t-SNE (may take a minute)...')
tsne = TSNE(n_components=2, perplexity=30, random_state=0)
g_2d = tsne.fit_transform(all_g)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Color by episode (different body params)
ax = axes[0, 0]
scatter = ax.scatter(g_2d[:, 0], g_2d[:, 1], c=all_ep, cmap='tab10', s=3, alpha=0.5)
plt.colorbar(scatter, ax=ax, label='Episode')
ax.set_title('g-space by episode (different bodies)')

# Color by time step
ax = axes[0, 1]
scatter = ax.scatter(g_2d[:, 0], g_2d[:, 1], c=all_t, cmap='viridis', s=3, alpha=0.5)
plt.colorbar(scatter, ax=ax, label='Time step')
ax.set_title('g-space by time within episode')

# Color by body height (z_pos) — a key dynamical variable
ax = axes[1, 0]
scatter = ax.scatter(g_2d[:, 0], g_2d[:, 1], c=all_obs[:, 0], cmap='coolwarm', s=3, alpha=0.5)
plt.colorbar(scatter, ax=ax, label='z_pos (height)')
ax.set_title('g-space by body height')

# Color by total body mass — if g captures structure, mass shouldn't cluster
ax = axes[1, 1]
scatter = ax.scatter(g_2d[:, 0], g_2d[:, 1], c=all_masses, cmap='plasma', s=3, alpha=0.5)
plt.colorbar(scatter, ax=ax, label='Total mass')
ax.set_title('g-space by body mass\n(overlap = structural abstraction)')

for ax in axes.flat:
    ax.set_xlabel('t-SNE 1')
    ax.set_ylabel('t-SNE 2')

plt.tight_layout()
plt.show()

### 4. Frequency Module Analysis
Do different frequency modules capture different timescales?

In [ ]:
# Collect single long episode for temporal analysis
cfg_fixed = cfg.copy()
cfg_fixed['randomize'] = False
env = DomainRandomizedHopper(cfg_fixed, seed=42)
obs, _ = env.reset()

n_steps_long = 400
chunk = []
for t in range(n_steps_long):
    action = env.action_space.sample()
    obs_normed = normalize_obs(obs)
    chunk.append({
        'obs': torch.tensor(obs_normed, dtype=torch.float).unsqueeze(0).to(device),
        'action': torch.tensor(action, dtype=torch.float).unsqueeze(0).to(device),
    })
    obs, _, term, trunc, _ = env.step(action)
    if term or trunc:
        obs, _ = env.reset()
env.close()

with torch.no_grad():
    steps_long = model(chunk)

n_f = cfg['n_f']
fig, axes = plt.subplots(n_f, 2, figsize=(14, 3.5 * n_f))

for f in range(n_f):
    g_f = np.stack([s.g_inf[f].squeeze(0).cpu().numpy() for s in steps_long])

    # Activations over time
    ax = axes[f, 0]
    n_show = min(6, g_f.shape[1])
    for neuron in range(n_show):
        ax.plot(g_f[:, neuron], alpha=0.7, linewidth=0.8)
    ax.set_title(f'Module {f} (alpha={cfg["f_initial"][f]:.2f}, n_g={cfg["n_g"][f]}): activations')
    ax.set_xlabel('Step')
    ax.set_ylabel('Activation')
    ax.grid(True, alpha=0.2)

    # Autocorrelation — should decay faster for high-freq modules
    ax = axes[f, 1]
    max_lag = min(100, len(g_f) // 2)
    mean_autocorr = np.zeros(max_lag)
    n_valid = 0
    for neuron in range(g_f.shape[1]):
        signal = g_f[:, neuron] - np.mean(g_f[:, neuron])
        norm = np.sum(signal ** 2)
        if norm > 1e-10:
            autocorr = np.correlate(signal, signal, mode='full')
            autocorr = autocorr[len(autocorr)//2 : len(autocorr)//2 + max_lag]
            mean_autocorr += autocorr / norm
            n_valid += 1
    if n_valid > 0:
        mean_autocorr /= n_valid
    ax.plot(mean_autocorr, color='steelblue', linewidth=2)
    ax.axhline(y=0, color='k', linestyle='--', alpha=0.3)
    ax.set_title(f'Module {f}: autocorrelation (slow decay = slow dynamics)')
    ax.set_xlabel('Lag (steps)')
    ax.set_ylabel('Autocorrelation')
    ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

### 5. Cross-Domain Transfer
Do similar dynamical states map to similar g-vectors across different body configurations?

In [ ]:
# PCA of g-space across 6 different body configs
n_configs = 6
config_g, config_height, config_angle, config_id = [], [], [], []

for c in range(n_configs):
    cfg_c = cfg.copy()
    cfg_c['randomize'] = True
    env = DomainRandomizedHopper(cfg_c, seed=9000 + c)
    obs, _ = env.reset()

    chunk = []
    raw_obs_list = []
    for t in range(200):
        action = env.action_space.sample()
        obs_normed = normalize_obs(obs)
        raw_obs_list.append(obs.copy())
        chunk.append({
            'obs': torch.tensor(obs_normed, dtype=torch.float).unsqueeze(0).to(device),
            'action': torch.tensor(action, dtype=torch.float).unsqueeze(0).to(device),
        })
        obs, _, term, trunc, _ = env.step(action)
        if term or trunc:
            obs, _ = env.reset()
    env.close()

    with torch.no_grad():
        steps_c = model(chunk)
    for i, step in enumerate(steps_c):
        config_g.append(torch.cat(step.g_inf, dim=1).squeeze(0).cpu().numpy())
        config_height.append(raw_obs_list[i][0])   # raw z_pos for interpretable axis
        config_angle.append(raw_obs_list[i][1])     # raw angle
        config_id.append(c)

config_g = np.stack(config_g)
config_height = np.array(config_height)
config_angle = np.array(config_angle)
config_id = np.array(config_id)

pca = PCA(n_components=2)
g_pca = pca.fit_transform(config_g)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

ax = axes[0]
for c in range(n_configs):
    mask = config_id == c
    ax.scatter(g_pca[mask, 0], g_pca[mask, 1], s=5, alpha=0.4, label=f'Body {c}')
ax.set_title('g-space: different body configs\n(overlap = generalization)')
ax.legend(fontsize=8, markerscale=3)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')

ax = axes[1]
scatter = ax.scatter(g_pca[:, 0], g_pca[:, 1], c=config_height, cmap='coolwarm', s=5, alpha=0.4)
plt.colorbar(scatter, ax=ax, label='Body height (z)')
ax.set_title('g-space colored by height\n(smooth gradient = structural encoding)')
ax.set_xlabel(f'PC1')
ax.set_ylabel(f'PC2')

ax = axes[2]
scatter = ax.scatter(g_pca[:, 0], g_pca[:, 1], c=config_angle, cmap='RdYlGn', s=5, alpha=0.4)
plt.colorbar(scatter, ax=ax, label='Body angle')
ax.set_title('g-space colored by body angle\n(smooth gradient = structural encoding)')
ax.set_xlabel(f'PC1')
ax.set_ylabel(f'PC2')

plt.tight_layout()
plt.show()

print(f'PCA explained variance: PC1={pca.explained_variance_ratio_[0]:.1%}, PC2={pca.explained_variance_ratio_[1]:.1%}')

### 6. Prediction Traces
Overlay predicted vs actual observations over time for a single episode.

In [ ]:
# Use the long episode from section 4
# Predictions and observations are in normalized space
obs_trace = np.stack([s.obs.squeeze(0).cpu().numpy() for s in steps_long])
pred_trace = np.stack([s.x_gen[0][0].squeeze(0).cpu().numpy() for s in steps_long])

dims_to_plot = [0, 1, 5, 7]  # z_pos, angle, x_vel, ang_vel
fig, axes = plt.subplots(len(dims_to_plot), 1, figsize=(14, 2.5 * len(dims_to_plot)), sharex=True)

for i, d in enumerate(dims_to_plot):
    ax = axes[i]
    ax.plot(obs_trace[:, d], color='black', linewidth=1, label='Actual', alpha=0.8)
    ax.plot(pred_trace[:, d], color='red', linewidth=1, label='Predicted', alpha=0.7, linestyle='--')
    ax.set_ylabel(f'{dim_names[d]} (norm.)')
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(True, alpha=0.2)
    mse = np.mean((obs_trace[:, d] - pred_trace[:, d]) ** 2)
    ax.set_title(f'{dim_names[d]} (MSE={mse:.4f})', fontsize=10)

axes[-1].set_xlabel('Step')
plt.suptitle('Predicted vs Actual Observations Over Time (normalized space)', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

### 7. Hebbian Memory Strength
How does the memory build up during an episode?

In [ ]:
# Track memory matrix norm over time
M_norms = [torch.norm(s.M[0]).cpu().item() for s in steps_long]
M_max = [torch.max(torch.abs(s.M[0])).cpu().item() for s in steps_long]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.plot(M_norms, color='steelblue', linewidth=1.5)
ax.set_xlabel('Step')
ax.set_ylabel('Frobenius norm')
ax.set_title('Hebbian Memory Strength Over Episode')
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(M_max, color='coral', linewidth=1.5)
ax.set_xlabel('Step')
ax.set_ylabel('Max |M_ij|')
ax.set_title('Hebbian Memory Max Entry')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Download results
import glob, os
run_dirs = sorted(glob.glob('runs/*/models'))
if run_dirs:
    latest = os.path.dirname(run_dirs[-1])
    print(f"Latest run: {latest}")
    !zip -r results.zip {latest}
    from google.colab import files
    files.download('results.zip')
else:
    print("No runs found yet.")